# Subject 3 · What speeches say: frequencies, keywords, concordance and named entities

**Course "Artificial Intelligence"** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Mini-project, subject 3** (text analysis for information extraction from political speeches) · **Chapter 2** (reading and processing text files: frequencies, concordance) · **Duration:** about 1 h 30

**What you will do**
- count the words of three excerpts from famous speeches by Abraham Lincoln and Woodrow Wilson, then find their most frequent words once stop words are set aside;
- compare the excerpts with **TF-IDF** to find the words that characterize each one, then go back to the text with a **concordance** (KWIC);
- automatically find places, people and organizations (**named entities**) with spaCy, and judge its mistakes.

**How to use this notebook**
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu "Runtime → Run all" runs the whole notebook at once. In Jupyter, these fields look like ordinary lines of code: change only the value after the = sign.

Before each code cell, a question asks you to **predict** the result: write down your answer, run the cell, then compare with the section "What you should see".

> **The texts.** Three short excerpts (under 150 words each) from public-domain speeches. Cuts are marked with […]; punctuation varies slightly from one edition to another.
>
> **Internet connection.** Only the cells of step 10 (named entities) download a model: they work in Colab. Everything else works offline.

## Step 1 · Load the tools

This cell loads the course tools: `re` (regular expressions, to split and search), `Counter` (to count), pandas (tables), matplotlib (charts) and, from scikit-learn, `TfidfVectorizer`, which calculates TF-IDF scores.

**Before you run it:** will this cell display a table, a chart, or only a short message?

In [ ]:
#@title Load the tools (libraries)
import re                                   # regular expressions: split, search
from collections import Counter             # count the items of a list
import pandas as pd                         # data tables
import matplotlib.pyplot as plt             # charts
from sklearn.feature_extraction.text import TfidfVectorizer   # TF-IDF scores
print("Tools loaded.")

**What you should see:** a single message, `Tools loaded.`

**Question:** which of the tools is used for counting? In which lesson of chapter 2 did you already meet it?

## Step 2 · The corpus: three speech excerpts

A **corpus** is a set of texts gathered for a research question. Ours has three excerpts, each stored in a variable (`extract_1`, `extract_2`, `extract_3`). Triple quotes `"""` let you write a text over several lines.

**Extract 1 · Abraham Lincoln, the Gettysburg Address**, Gettysburg, Pennsylvania, 19 November 1863. During the Civil War, Lincoln dedicates a cemetery for the soldiers killed in the battle of Gettysburg (text of the "Bliss copy").

**Before you run it:** the cell counts the words by cutting the text at the spaces. How many words, in your opinion: about 50, 130 or 300?

In [ ]:
#@title Extract 1: Abraham Lincoln, the Gettysburg Address (1863)
extract_1 = """Four score and seven years ago our fathers brought forth on this continent, a new
nation, conceived in Liberty, and dedicated to the proposition that all men are
created equal. Now we are engaged in a great civil war, testing whether that nation,
or any nation so conceived and so dedicated, can long endure. We are met on a great
battle-field of that war. We have come to dedicate a portion of that field, as a final
resting place for those who here gave their lives that that nation might live. It is
altogether fitting and proper that we should do this. […] that this nation, under God,
shall have a new birth of freedom—and that government of the people, by the people,
for the people, shall not perish from the earth."""
print("Extract 1:", len(extract_1.split()), "words (cutting at the spaces)")

**What you should see:** `Extract 1: 132 words (cutting at the spaces)`.

**Question:** Lincoln ends with "of the people, by the people, for the people". What is the effect of repeating the same noun with three different prepositions?

**Extract 2 · Abraham Lincoln, First Inaugural Address**, Washington, D.C., 4 March 1861. On the day he becomes president, seven Southern states have already left the Union; the Civil War begins five weeks later.

**Before you run it:** read the text of the cell. Which pronouns does Lincoln use to address the Southern states, and which to speak of himself?

In [ ]:
#@title Extract 2: Abraham Lincoln, First Inaugural Address (1861)
extract_2 = """In your hands, my dissatisfied fellow-countrymen, and not in mine, is the momentous
issue of civil war. The Government will not assail you. You can have no conflict
without being yourselves the aggressors. You have no oath registered in heaven to
destroy the Government, while I shall have the most solemn one to "preserve, protect,
and defend it." I am loath to close. We are not enemies, but friends. We must not be
enemies. Though passion may have strained it must not break our bonds of affection.
The mystic chords of memory, stretching from every battlefield and patriot grave to
every living heart and hearthstone all over this broad land, will yet swell the
chorus of the Union, when again touched, as surely they will be, by the better angels
of our nature."""
print("Extract 2:", len(extract_2.split()), "words (cutting at the spaces)")

**What you should see:** `Extract 2: 133 words (cutting at the spaces)`. Lincoln addresses the South as "you" ("In your hands", "The Government will not assail you"), speaks of himself as "I", then moves to "we" and "our" at the end.

**Question:** "We are not enemies, but friends. We must not be enemies." What changes between "are not" and "must not"?

**Extract 3 · Woodrow Wilson, the "Fourteen Points" address**, Congress, Washington, D.C., 8 January 1918. During the First World War, Wilson sets out fourteen conditions for a lasting peace. The excerpt gives points VII, VIII and IX; their numbers have been removed so that they are not counted as words.

**Before you run it:** how many names of countries can you count in the text of the cell?

In [ ]:
#@title Extract 3: Woodrow Wilson, the Fourteen Points (1918)
extract_3 = """Belgium, the whole world will agree, must be evacuated and restored, without any
attempt to limit the sovereignty which she enjoys in common with all other free
nations. […] All French territory should be freed and the invaded portions restored,
and the wrong done to France by Prussia in 1871 in the matter of Alsace-Lorraine,
which has unsettled the peace of the world for nearly fifty years, should be righted,
in order that peace may once more be made secure in the interest of all. A
readjustment of the frontiers of Italy should be effected along clearly recognizable
lines of nationality."""
print("Extract 3:", len(extract_3.split()), "words (cutting at the spaces)")

**What you should see:** `Extract 3: 101 words (cutting at the spaces)`. There are four countries (Belgium, France, Prussia, Italy), one region (Alsace-Lorraine), one adjective of nationality (French) and one date (1871). Keep this list: in step 10, spaCy will have to find it.

**Question:** Wilson writes that Belgium "must" be evacuated, but that French territory and the frontiers of Italy "should" be restored and readjusted. Does the choice of modal verb make a difference?

## Step 3 · Gather the corpus and its record sheet

The mini-project asks you to note, for each speech, the speaker, the date, the place and the language, and to state whether it is the original or a translation. This cell stores these **metadata** in a pandas table, with one row per excerpt.

**Before you run it:** how many rows and how many columns will this table have?

In [ ]:
#@title The record sheet of the corpus (metadata)
records = pd.DataFrame({
    "speaker": ["Abraham Lincoln", "Abraham Lincoln", "Woodrow Wilson"],
    "date": ["19 November 1863", "4 March 1861", "8 January 1918"],
    "place": ["Gettysburg, cemetery", "Washington, Capitol", "Washington, Congress"],
    "language": ["English (original)", "English (original)", "English (original)"],
}, index=["Lincoln 1863, Gettysburg", "Lincoln 1861, inaugural", "Wilson 1918, Fourteen Points"])
records

**What you should see:** a table with 3 rows and 4 columns (speaker, date, place, language). The short row titles ("Lincoln 1863, Gettysburg"…) will be used as labels in the rest of the notebook.

**Question:** why is it important, in a study of speeches, to know whether you are working on the original or on a translation? Think about word counts.

The next cell builds the **corpus**: a dictionary that links each title to the text of the excerpt. It also lets you add a fourth excerpt if you wish (a short passage from a public-domain speech, or a paragraph you write yourself). **For now, leave the field `my_extract` empty.**

**Before you run it:** if the field stays empty, how many excerpts will the program announce?

In [ ]:
#@title TO CHANGE (optional): add your own excerpt to the corpus
my_title = "My extract" #@param {type:"string"}
my_extract = "" #@param {type:"string"}
corpus = dict(zip(records.index, [extract_1, extract_2, extract_3]))   # title -> text
if my_extract.strip() != "":             # the field is not empty
    corpus[my_title] = my_extract         # we add a 4th excerpt
print(len(corpus), "excerpts in the corpus:")
for title in corpus:
    print("  -", title)

**What you should see:**
```
3 excerpts in the corpus:
  - Lincoln 1863, Gettysburg
  - Lincoln 1861, inaugural
  - Wilson 1918, Fourteen Points
```
**Question:** three excerpts of about a hundred words make a very small corpus. How many speeches would you need, in your opinion, to answer the research question of the mini-project (the themes of the UN general debate)?

## Step 4 · Split into words and count

Cutting at the spaces is approximate: "[…]" becomes a "word", and "freedom—and" stays one word. The function `words_of` uses the regular expression `\w+` instead ("a sequence of letters or digits"), as in chapter 2, after putting everything in lower case. The table then counts, for each excerpt:
- the **words** (occurrences, *tokens*): every appearance counts;
- the **different forms** (*types*): "nation" written five times counts only once;
- the ratio **types / tokens**, a simple measure of the variety of the vocabulary.

The cell also displays the first 12 words of Lincoln's inaugural as Python splits them.

**Before you run it:** the inaugural had 133 words when cut at the spaces. Will the `\w+` split find more or fewer? Look at "fellow-countrymen".

In [ ]:
#@title Split each excerpt into words, then count tokens and types
def words_of(text):
    """List of the words of a text, in lower case (sequences of letters or digits)."""
    return re.findall(r"\w+", text.lower())

print(words_of(extract_2)[:12])           # the first 12 words of the inaugural
sizes = pd.DataFrame(index=list(corpus))
sizes["tokens"] = [len(words_of(text)) for text in corpus.values()]
sizes["types"] = [len(set(words_of(text))) for text in corpus.values()]
sizes["types / tokens"] = (sizes["types"] / sizes["tokens"]).round(2)
sizes

**What you should see:** the list `['in', 'your', 'hands', 'my', 'dissatisfied', 'fellow', 'countrymen', 'and', 'not', 'in', 'mine', 'is']`, then this table:

| | tokens | types | types / tokens |
|---|---|---|---|
| Lincoln 1863, Gettysburg | 133 | 83 | 0.62 |
| Lincoln 1861, inaugural | 134 | 93 | 0.69 |
| Wilson 1918, Fourteen Points | 101 | 69 | 0.68 |

The hyphen cuts "fellow-countrymen" into `fellow` + `countrymen`: the inaugural goes from 133 to 134 words. Note also that the Gettysburg text writes "battle-field" (two words for Python) and the inaugural "battlefield" (one word): spelling choices change the counts.

**Question:** Gettysburg has the lowest types / tokens ratio (0.62). Which repetitions, noticed in step 2, partly explain it?

## Step 5 · The most frequent words of the whole corpus

The cell gathers the words of the three excerpts into one list, then `Counter` counts each form and `most_common(10)` gives the ten most frequent ones, as (word, number) pairs.

**Before you run it:** write down the three words that you think will come first. Do words like "nation", "war" or "peace" stand a chance?

In [ ]:
#@title The 10 most frequent words of the whole corpus
all_words = []
for text in corpus.values():
    all_words = all_words + words_of(text)            # add the words of each excerpt
count = Counter(all_words)
print(len(all_words), "words in all,", len(count), "different forms")
print(count.most_common(10))

**What you should see:**
```
368 words in all, 205 different forms
[('the', 23), ('of', 15), ('and', 12), ('in', 10), ('that', 10), ('to', 8), ('a', 7), ('be', 7), ('we', 6), ('have', 6)]
```
The top ten places go to **stop words**: articles, prepositions, conjunctions, pronouns and auxiliaries. They are found in every text and say nothing about the subject.

**Question:** are these words really useless for everybody? Think of a teacher studying the use of articles, or of a stylistician studying the speaker's "we".

## Step 6 · Set stop words aside

A **stop-word list** is a list of function words to ignore. Ours, written directly in the cell, has 75 words. You can add more in the field `extra_stop_words`, separated by commas. **For now, leave it empty.**

**Before you run it:** what will the cell display, if the field stays empty?

In [ ]:
#@title TO CHANGE: the stop-word list (function words to ignore)
extra_stop_words = "" #@param {type:"string"}
stop_words = set("""the a an and or but nor of to in on at by for with from as into over under
up is are was were be been being am it its this that these those which who whom
what there their they them we our us you your yourselves i me my mine he him his
she her not no so all any here than then if when while do does did have has had""".split())
for word in extra_stop_words.replace(",", " ").lower().split():
    stop_words.add(word)                  # add the words of the field
print(len(stop_words), "stop words in the list.")

**What you should see:** `75 stop words in the list.` If you add words in the field, the number increases accordingly.

**Question:** the list contains `not` and `no`. What do we lose by setting negation aside, in a speech like Lincoln's inaugural ("We are not enemies")?

The next cell keeps, in each excerpt, the **content words** (the words that are not in the stop-word list), then displays the eight most frequent ones of each excerpt, with their number of occurrences in brackets. The rows of the table are numbered from 0, as always in Python.

**Before you run it:** which word will come first in the Gettysburg Address? And in Wilson's excerpt?

In [ ]:
#@title The 8 most frequent content words of each excerpt
ranking = {}
for title, text in corpus.items():
    content = [w for w in words_of(text) if w not in stop_words]   # remove the stop words
    first = Counter(content).most_common(8)
    ranking[title] = pd.Series([f"{word} ({n})" for word, n in first])
pd.DataFrame(ranking)

**What you should see:**

| | Lincoln 1863, Gettysburg | Lincoln 1861, inaugural | Wilson 1918, Fourteen Points |
|---|---|---|---|
| 0 | nation (5) | will (3) | should (3) |
| 1 | people (3) | government (2) | world (2) |
| 2 | new (2) | enemies (2) | restored (2) |
| 3 | conceived (2) | must (2) | peace (2) |
| 4 | dedicated (2) | every (2) | belgium (1) |
| 5 | great (2) | hands (1) | whole (1) |
| 6 | war (2) | dissatisfied (1) | will (1) |
| 7 | field (2) | fellow (1) | agree (1) |

Content words reveal the theme ("nation", "peace") and the rhetoric: Lincoln's "the people" repeated three times, Wilson's "should". Modal verbs (will, must, should) are not in our stop-word list, so they appear: they express obligation, prediction or recommendation. When two words are tied, `Counter` keeps their order of appearance in the text.

**Question:** Wilson's top word is "should", Lincoln's inaugural has "will" and "must". What does this tell you about the tone of each speech?

## Step 7 · A bar chart

A bar chart makes the ranking readable at a glance. Choose the excerpt in the drop-down list and the number of words with the slider: the cell counts the content words of this excerpt and draws one bar per word, the longest at the top, with its number at the end.

**Before you run it:** for the excerpt chosen by default (Gettysburg), how many bars will have exactly the same length as the bar of "new"?

In [ ]:
#@title TO CHANGE: the excerpt and the number of words in the chart
chosen_extract = "Lincoln 1863, Gettysburg" #@param ["Lincoln 1863, Gettysburg", "Lincoln 1861, inaugural", "Wilson 1918, Fourteen Points"] {allow-input: true}
number_of_words = 10 #@param {type:"slider", min:5, max:20, step:1}
content = [w for w in words_of(corpus[chosen_extract]) if w not in stop_words]
first = Counter(content).most_common(number_of_words)      # (word, number) pairs
words = [word for word, n in first]
numbers = [n for word, n in first]
fig, ax = plt.subplots(figsize=(7, 1 + 0.35 * len(words)))
bars = ax.barh(words, numbers, color="#2a78d6", height=0.6)
ax.bar_label(bars, padding=3)              # the number at the end of each bar
ax.invert_yaxis()                          # the most frequent word at the top
ax.set_xlabel("number of occurrences")
ax.set_title(f"{chosen_extract}: the most frequent content words")
plt.show()

**What you should see:** ten blue bars. `nation` at the top (5), then `people` (3), then seven words at 2: `new`, `conceived`, `dedicated`, `great`, `war`, `field`, `shall`, and finally `four` (1). Six other bars therefore have the same length as the bar of "new".

**Question:** with so few levels (5, 3, 2, 1), does the chart really tell the words apart? Choose "Lincoln 1861, inaugural" and 15 words: what do you notice from the sixth word on?

## Step 8 · Compare the excerpts with TF-IDF

Frequencies describe each excerpt on its own. **TF-IDF** compares: a word is **characteristic** of an excerpt if it is frequent there **and** rare in the others.
- **TF** (*term frequency*): the number of occurrences of the word in the excerpt.
- **IDF** (*inverse document frequency*): the "rarity" of the word in the corpus. With the scikit-learn formula, IDF = ln((1 + N) / (1 + n)) + 1, where N is the number of excerpts and n the number of excerpts that contain the word. For N = 3: a word found in all 3 excerpts gets 1.00, in 2 excerpts 1.29, in only one 1.69.
- The TF-IDF score is the product TF × IDF; scikit-learn then brings it to a common scale, so that long and short excerpts stay comparable.

The first cell calculates the IDF of a few words, keeping all the words here, stop words included.

**Before you run it:** rank `the`, `war` and `nation` from the "rarest" to the most "common" in the corpus.

In [ ]:
#@title TO CHANGE: the "rarity" (IDF) of a few words
words_to_examine = "the, war, nation" #@param {type:"string"}
everything = TfidfVectorizer(token_pattern=r"\w+").fit(corpus.values())   # keep all words here
idf = pd.Series(everything.idf_, index=everything.get_feature_names_out())
for word in words_to_examine.replace(",", " ").lower().split():
    present = sum(1 for text in corpus.values() if word in words_of(text))    # nb of excerpts
    if word in idf.index:
        print(f"{word:<10} in {present} excerpt(s) out of {len(corpus)}   IDF = {idf[word]:.2f}")
    else:
        print(f"{word:<10} not in the corpus")

**What you should see:**
```
the        in 3 excerpt(s) out of 3   IDF = 1.00
war        in 2 excerpt(s) out of 3   IDF = 1.29
nation     in 1 excerpt(s) out of 3   IDF = 1.69
```
`war` appears in both Lincoln speeches ("a great civil war", "the momentous issue of civil war"), `nation` only at Gettysburg (Wilson says "nations", which Python counts as another word).

**Question:** in a corpus of 50 speeches, a word found in a single speech would get an IDF of ln(51 / 2) + 1, about 4.2. Why does TF-IDF become more useful as the corpus grows?

The next cell calculates the TF-IDF scores of all the words of each excerpt, then displays the eight most characteristic words of each one, with their score in brackets. The box `remove_stop_words` (ticked) applies our stop-word list.

**Before you run it:** at Gettysburg, `war` appeared twice (step 6). Will it stay among the eight most characteristic words? Use the previous result to decide.

In [ ]:
#@title The 8 most characteristic words of each excerpt (TF-IDF)
remove_stop_words = True #@param {type:"boolean"}
if remove_stop_words:
    vectorizer = TfidfVectorizer(token_pattern=r"\w+", stop_words=sorted(stop_words))
else:
    vectorizer = TfidfVectorizer(token_pattern=r"\w+")
matrix = vectorizer.fit_transform(corpus.values())       # one row per excerpt, one column per word
scores = pd.DataFrame(matrix.toarray(), index=list(corpus), columns=vectorizer.get_feature_names_out())
characteristic = {}
for title in corpus:
    best = scores.loc[title].sort_values(ascending=False, kind="stable").head(8)
    characteristic[title] = [f"{word} ({s:.2f})" for word, s in best.items()]
pd.DataFrame(characteristic)

**What you should see:**

| | Lincoln 1863, Gettysburg | Lincoln 1861, inaugural | Wilson 1918, Fourteen Points |
|---|---|---|---|
| 0 | nation (0.50) | will (0.27) | should (0.28) |
| 1 | people (0.30) | enemies (0.24) | peace (0.25) |
| 2 | conceived (0.20) | every (0.24) | restored (0.25) |
| 3 | dedicated (0.20) | government (0.18) | world (0.25) |
| 4 | field (0.20) | must (0.18) | agree (0.12) |
| 5 | great (0.20) | affection (0.12) | alsace (0.12) |
| 6 | new (0.20) | aggressors (0.12) | attempt (0.12) |
| 7 | shall (0.15) | angels (0.12) | belgium (0.12) |

`war` has left the Gettysburg list: it appears twice, like `new`, but it is shared with the inaugural (IDF 1.29 instead of 1.69). `shall` goes down for the same reason. When two scores are equal, the words are in alphabetical order. Untick the box and run again: without a stop-word list, `that`, `the`, `a` come back to the top, because with three excerpts IDF can only divide the weight of a common word by 1.69 at most.

**Question:** according to this table, which three words would best sum up each excerpt? Are these words themes, rhetorical devices, or both?

## Step 9 · Back to the text: the concordance (KWIC)

A word counted out of its context can be misleading. The KWIC concordance (*keyword in context*) displays each occurrence of a word in the middle of its left and right context, all aligned on the word searched for: it is the `kwic` function of chapter 2, adapted to prose. We add a convenience, as in the AntConc or TXM software: the star `*` stands for any beginning or end of a word (`nation*` finds nation, nations, nationality).

This cell **defines** the function, then tries it on extract 1 with the word "nation".

**Before you run it:** how many lines will the concordance of "nation" display?

In [ ]:
#@title Define the concordance function (KWIC) and try it
def kwic(text, word, width=35):
    """Each occurrence of word, with its left and right context (* = any letters)."""
    flat = " ".join(text.split())                           # the whole excerpt on one line
    pattern = r"\b" + word.replace("*", r"\w*") + r"\b"     # nation* becomes nation\w*
    lines = []
    for m in re.finditer(pattern, flat, flags=re.IGNORECASE):
        left = flat[max(0, m.start() - width):m.start()]
        right = flat[m.end():m.end() + width]
        lines.append(f"{left:>{width}}[{m.group()}]{right}")
    return lines

for line in kwic(extract_1, "nation"):
    print(line)

**What you should see:** five lines aligned on the word in square brackets.

**Question:** read the left-hand column. Which small word comes just before "nation" almost every time? What does the concordance show that the frequency (5) did not?

Now search the **whole corpus**. Type a word (or the beginning of a word followed by `*`) in the field `search_word`; the slider `width` sets the number of characters of context on each side.

**Before you run it:** the word `war` appears in two excerpts. Which adjective will you find just before it?

In [ ]:
#@title TO CHANGE: the word to search for in the whole corpus
search_word = "war" #@param {type:"string"}
width = 35 #@param {type:"slider", min:20, max:60, step:5}
for title, text in corpus.items():
    concordance = kwic(text, search_word, width)
    print(f"--- {title}: {len(concordance)} occurrence(s)")
    for line in concordance:
        print(line)

**What you should see:** (see exact output)

**Question:** ...

## Step 10 · Named entities with spaCy (online)

A **named entity** refers to a specific being or object: a person, a place, an organization, a date. Named entity recognition is a **machine learning** task: spaCy's `en_core_web_sm` model has learned to find them from thousands of annotated sentences. Its main labels:

| Label | Meaning | Example |
|---|---|---|
| `PERSON` | person | Karim Benali |
| `GPE` | country, city, state | France, Algiers |
| `LOC` | other place (mountain, river, region) | the Sahara |
| `ORG` | organization | UNESCO |
| `NORP` | nationality, religious or political group | French |
| `DATE` | date or period | 1871, nearly fifty years |
| `CARDINAL` | number | four |

> **Needs an internet connection; works in Colab.** The four cells of this step download spaCy and its English model (about 13 MB). Offline, simply read the expected results. No cell after them depends on them.

The first cell installs spaCy (already present in Colab) and downloads the model.

**Before you run it:** will this cell analyze our excerpts?

In [ ]:
#@title Install spaCy and download the English model (internet connection)
%pip install -q spacy
!python -m spacy download en_core_web_sm

**What you should see:** download lines, then `✔ Download and installation successful` and `You can now load the package via spacy.load('en_core_web_sm')`. Nothing is analyzed yet. If the next cell says that the model cannot be found, choose "Runtime → Restart session", then run the whole notebook again.

**Question:** why must a machine learning model be downloaded, whereas our stop-word list fitted in the cell?

The next cell loads the model, analyzes each excerpt and stores each entity found in a table: the excerpt, the text of the entity and its type.

**Before you run it:** take your list from step 2 (four countries, one region, one nationality and one date in Wilson's excerpt). Will spaCy find them all? Will it find entities in the inaugural?

In [ ]:
#@title Find the named entities of each excerpt (internet connection)
import spacy
nlp = spacy.load("en_core_web_sm")                 # spaCy's small English model
rows = []                                           # one row per entity found
for title, text in corpus.items():
    doc = nlp(" ".join(text.split()))               # the excerpt on one line
    for ent in doc.ents:                            # ent.text: the entity; ent.label_: its type
        rows.append({"extract": title, "entity": ent.text, "type": ent.label_})
entities = pd.DataFrame(rows)
entities

**What you should see:** (NER1)

The mini-project asks for a **table of entities** per speech. The next cell counts the entities by type (in rows) and by excerpt (in columns). Excerpts with no entity at all are kept, with zeros.

**Before you run it:** which cell of the table will have the largest number?

In [ ]:
#@title Count the entities by type and by excerpt (internet connection)
entity_table = pd.crosstab(entities["type"], entities["extract"])
entity_table = entity_table.reindex(columns=list(corpus), fill_value=0)   # keep every excerpt
entity_table

**What you should see:** (NER2)

Finally, test spaCy on a sentence of your choice. The sentence given is **made up** for the example (adapted from the mini-project lesson).

**Before you run it:** which entities, and of which types, should spaCy find in this sentence?

In [ ]:
#@title TO CHANGE: test spaCy on a sentence (internet connection)
sentence = "In Constantine, the minister Karim Benali discussed the renovation of the medina with UNESCO on 12 March 2024." #@param {type:"string"}
for ent in nlp(sentence).ents:
    print(ent.text, "->", ent.label_)

**What you should see:** (NER3)

## Your turn

Change only the fields, then run the cells again from the one you changed (in Colab: "Runtime → Run after", or "Runtime → Run all").
1. **Step 9, a family of words:** type `nation*` in `search_word`. The concordance brings together "nation", "nations" and "nationality": who uses which, and about what? Try also `peace`, `people`, `free*`.
2. **Step 6, stop words:** add the modal verbs `will, must, should, shall, may, can` to `extra_stop_words`, then run steps 6 to 8 again. What changes in the chart and in the TF-IDF table? Are modal verbs really "empty" in a political speech?
3. **Step 3, a fourth excerpt:** paste into `my_extract` a short passage (about a hundred words) of a public-domain speech you are studying, or a paragraph you write yourself about war or the nation, then run the whole notebook again. Which of Lincoln's words lose their status as characteristic words?

## For the mini-project: your own corpus

For your mini-project, replace our three excerpts with **your** corpus:
- prepare a CSV file (in a spreadsheet: "Save as… → CSV UTF-8") with **one row per speech** and at least two columns, `title` and `text`; add the metadata required (`speaker`, `date`, `place`, `language`, original or translation);
- clean the texts: remove what is not part of the speech ("applause", headers, page numbers);
- tick the box below and run the cell: in Colab, a window asks you for the file; in Jupyter, put the file next to the notebook under the name `my_speeches.csv`.

**Before you run it:** if you leave the box unticked, which corpus will the cell use?

In [ ]:
#@title TO CHANGE: load your own corpus (CSV file, columns title and text)
load_my_file = False #@param {type:"boolean"}
my_speeches = pd.DataFrame({"title": list(corpus), "text": list(corpus.values())})  # by default
if load_my_file:
    try:
        from google.colab import files       # only exists in Colab
        upload = files.upload()              # window to choose the file
        file_name = list(upload.keys())[0]
    except ImportError:                      # Jupyter: file placed next to the notebook
        file_name = "my_speeches.csv"
    my_speeches = pd.read_csv(file_name)
print("Corpus used:", len(my_speeches), "speeches")

**What you should see:** with the box unticked, `Corpus used: 3 speeches` (our excerpts). With your file, the number of rows of your table.

The next cell sums up each speech of the loaded corpus: its size, its most frequent content words and its most characteristic words according to TF-IDF.

**Before you run it:** for our excerpts, will you find words already seen in steps 6 and 8?

In [ ]:
#@title Sum up each speech: size, frequent words, characteristic words
vect = TfidfVectorizer(token_pattern=r"\w+", stop_words=sorted(stop_words))
corpus_scores = pd.DataFrame(vect.fit_transform(my_speeches["text"]).toarray(),
                             columns=vect.get_feature_names_out())
for i in range(len(my_speeches)):
    title, text = my_speeches["title"][i], my_speeches["text"][i]
    content = [w for w in words_of(text) if w not in stop_words]
    best = corpus_scores.loc[i].sort_values(ascending=False, kind="stable").head(5)
    print(f"{title}: {len(words_of(text))} words, {len(set(words_of(text)))} types")
    print("    most frequent     :", [word for word, n in Counter(content).most_common(5)])
    print("    characteristic    :", list(best.index))

**What you should see:** (MP)

## To hand in

Answer in writing, in English, in a few lines for each question.
1. **Characteristic words.** Choose one excerpt. What are its three most characteristic words according to TF-IDF? Using the concordance (step 9), show how the speaker uses them: anaphora, lexical field, contrast…
2. **Counting or comparing.** For this excerpt, compare the ranking by frequency (step 6) and the TF-IDF ranking (step 8). What changes, and why? In what kind of corpus would TF-IDF be even more useful?
3. **Judging the tool.** Note two mistakes or two omissions made by spaCy in step 10 (if you could not run it, use the expected results). What do they teach you about how far you can trust an automatic tool in your mini-project, and how would you check it?

## Key points

- Counting **tokens** and **types** means deciding what a word is (spaces, apostrophes, hyphens); raw frequencies are dominated by **stop words**, and the stop-word list is an analytical choice.
- **TF-IDF** compares texts: it highlights the words that are frequent in one text and rare in the others. The larger the corpus, the more useful it is; the **concordance** (KWIC) always takes you back from the figures to the text.
- **Named entity recognition** (spaCy) finds places, people and organizations through machine learning, but it makes mistakes: always check it on a sample.

**Lessons on the course site:** [Mini-project, subject 3: political speeches](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-3) · [Chapter 2: reading and processing text files (frequencies, concordance)](https://progremer04.github.io/canva-au-for-ang-/en/#c2-fichiers)